### Open Field Ephys QC and Summary

1) Treats "mouse_name" as MouseID and builds a per-mouse summary table with counts and mean/median metrics.
2) Displays the first 5 rows and the per-mouse summary.
3) Plots mean_firing_rate distributions by mouse (boxplot).
4) Plots spatial_info vs coherence, with points grouped and colored by mouse, and legend outside.



Load the linear_track.xlsx data into a DataFrame, cache it as Parquet, reload it, and preview basic info (head/shape/columns).
Perform data quality checks on df (trim column names, missing values, duplicates, outliers, type consistency, and categorical value consistency).

"""
- Load actogram data from Excel- Convert to Parquet for faster loading
- Basic QC checks on the dataframe structure and content
  - Check for leading/trailing whitespace in column names
  - Count missing values per column
  - Count duplicate rows
  - Identify outliers in numeric columns (using IQR method)
  - Check for inconsistent data types within columns
  - List unique values in categorical columns
- Basic sanity checks on the data content
    - Distribution of mean firing rates across cells and mice
    - Relationship between spatial information and coherence
    - Per-mouse summary statistics (number of cells, mean firing rate, etc.)
- Violin plots with overlaid points (one colour per mouse, using tab20b)
    - Mean and median lines styled differently from default (e.g. mean in green, median in black)
    - Legend for mean/median line styles (not from tab20/tab20b)
    - Points use tab20b, one colour per mouse
- Statistical comparisons between experimenters (Mann–Whitney U test)
    - Only if there are exactly 2 experimenters and at least 3 samples per group
    - Annotate plots with p-values and significance stars
    - Include summary statistics (n, mean, median) in the annotation
"""


In [4]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime as dt
import re, math, textwrap
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.lines import Line2D

# Optional: nice dataframe display in notebooks (falls back to print)
try:
    from IPython.display import display  # type: ignore
    def display_df(title, x):
        print(title)
        display(x)
except Exception:
    def display_df(title, x):
        print(title)
        print(x)

# Set the input path ONCE (edit this one line only)
linear_track_XLSX_PATH = "/Users/loukia/UCL Dropbox/Loukia Katsouri/DataProtocolsEquipment/Ephys_Analysis/RobinData/csv_files/linear_track.xlsx"
linear_track_DIR = os.path.dirname(linear_track_XLSX_PATH)

df_linear_track = pd.read_excel(linear_track_XLSX_PATH)

PARQUET_PATH = "linear_track.parquet"
df_linear_track.to_parquet(PARQUET_PATH, index=False)
df_linear_track = pd.read_parquet(PARQUET_PATH)

df_linear_track.head(), df_linear_track.shape, df_linear_track.columns.tolist()[:5]

(                                            filename  tetrode  cluster  \
 0  /home/robin/Documents/Science/SWC/jok_loukia_r...        1        1   
 1  /home/robin/Documents/Science/SWC/jok_loukia_r...        1        2   
 2  /home/robin/Documents/Science/SWC/jok_loukia_r...        1        3   
 3  /home/robin/Documents/Science/SWC/jok_loukia_r...        1        4   
 4  /home/robin/Documents/Science/SWC/jok_loukia_r...        1        5   
 
      trial_type  phase_locking_vector_length  phase_locking_z_stat  \
 0  lineartracka                     0.014087              2.440816   
 1  lineartracka                     0.063604             13.050800   
 2  lineartracka                     0.239017             10.340374   
 3  lineartracka                     0.002509              0.050968   
 4  lineartracka                     0.139044              7.481996   
 
    phase_locking_pval  trial_type_x  phase_mean  phase_variance  ...  \
 0            0.087088  lineartracka    1.42393

In [5]:
"""Basic QC checks on the dataframe structure and content
- Check for leading/trailing whitespace in column names
- Count missing values per column
- Count duplicate rows
- Identify outliers in numeric columns (using IQR method)
- Check for inconsistent data types within columns
- List unique values in categorical columns
"""

# Remove leading/trailing whitespace from column names
df_linear_track.columns = df_linear_track.columns.str.strip()

# Missing values per column
missing_values = df_linear_track.isnull().sum()
print("Missing values in each column:")
print(missing_values)

# Duplicate rows
duplicate_rows = df_linear_track.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_rows}")

# Outliers in numeric columns via IQR
numerical_cols = df_linear_track.select_dtypes(include=[np.number]).columns
for col in numerical_cols:
    Q1 = df_linear_track[col].quantile(0.25)
    Q3 = df_linear_track[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df_linear_track[(df_linear_track[col] < lower_bound) | (df_linear_track[col] > upper_bound)]
    print(f"Column: {col}, Number of outliers: {len(outliers)}")

# Inconsistent types per column (can be slow on very large dataframes)
for col in df_linear_track.columns:
    data_types = df_linear_track[col].apply(type).value_counts()
    print(f"Column: {col}, Data types:\n{data_types}\n")

# Unique values in categorical columns
categorical_cols = df_linear_track.select_dtypes(include=["object"]).columns
for col in categorical_cols:
    unique_values = df_linear_track[col].unique()
    print(f"Column: {col}, Unique values:\n{unique_values}\n")

Missing values in each column:
filename                         0
tetrode                          0
cluster                          0
trial_type                       0
phase_locking_vector_length      0
phase_locking_z_stat             0
phase_locking_pval               0
trial_type_x                     0
phase_mean                       0
phase_variance                   0
trial_type_y                     0
odd_even_stability               0
half_split_stability            14
firing_rate                      0
num_spikes                       0
spike_width                      0
peak_rate                        0
spatial_info                     0
coherence                        0
mouse_name                       0
spatial_info_left                4
spatial_info_right               6
overlap_score                    8
n_spikes_left                    0
n_spikes_right                   0
Phenotype                        0
Cell type                        0
Experimenter            

In [6]:
"""Basic sanity checks on the data content
- Distribution of mean firing rates across cells and mice
- Relationship between spatial information and coherence
- Per-mouse summary statistics (number of cells, mean firing rate, etc.)
"""

# Treat mouse_name as MouseID
mouse_col = "mouse_name"
mice = sorted(df_linear_track[mouse_col].dropna().unique().tolist())

# Per-mouse summary
summary = (df_linear_track.groupby(mouse_col)
             .agg(
                 n_cells=("cluster","count"),
                 n_tetrodes=("tetrode","nunique"),
                 mean_mean_firing_rate=("mean_firing_rate","mean"),
                 median_mean_firing_rate=("mean_firing_rate","median"),
                 mean_spatial_info=("spatial_info","mean"),
                 mean_coherence=("coherence","mean"),
                 mean_field_size=("field_size","mean"),
                 mean_theta_modulation=("theta_modulation","mean"),
             )
             .reset_index()
             .sort_values("n_cells", ascending=False))

display_df("Per-mouse summary", summary)
save_path = os.path.join(linear_track_DIR, "linear_track_summary.csv")
summary.to_csv(save_path, index=False)
print(f"Summary saved to: {save_path}")
print("-----------------------------------------------------------------------------------------------------------------------------------------------")                    
display_df("Open field data (first 100 rows)", df_linear_track.head(100))


# Plot 1: distribution of mean firing rate per mouse
plt.figure()
df_linear_track.boxplot(column="mean_firing_rate", by=mouse_col, grid=False, rot=90, showfliers=False)
plt.title("Mean firing rate by mouse")
plt.suptitle("")
plt.xlabel("MouseID (mouse_name)")
plt.ylabel("mean_firing_rate")
plt.tight_layout()
plt.show()

# Plot 2: spatial_info vs coherence (per cell), grouped by mouse with color + marker
plt.figure()

markers = ["o", "s", "^", "D", "v", "P", "X", "*", "<", ">", "h", "p"]
cmap = plt.get_cmap("tab20")  # 20 distinct colors, then repeats

for i, m in enumerate(mice):
    g = df_linear_track.loc[df_linear_track[mouse_col] == m]
    plt.scatter(
        g["coherence"],
        g["spatial_info"],
        label=str(m),
        alpha=0.5,
        s=20,
        marker=markers[i % len(markers)],
        color=cmap(i % 20),
    )

plt.title("Spatial info vs coherence (per cell), grouped by mouse")
plt.xlabel("coherence")
plt.ylabel("spatial_info")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0, fontsize=8)
plt.tight_layout()
plt.show()

KeyError: "Column(s) ['field_size', 'mean_firing_rate', 'theta_modulation'] do not exist"

### Generate boxplots with overlaid points for all numeric columns, grouped by mouse.
#### Create a multi-page PDF with one plot per numeric column showing distributions across mice.

In [ ]:
mouse_col = "mouse_name"
mice = sorted(df_linear_track[mouse_col].dropna().unique().tolist())
numeric_cols = [c for c in df_linear_track.select_dtypes(include=[np.number]).columns if c != mouse_col]

rng = np.random.default_rng(0)

def boxplot_with_points(ax, data, col, mice, jitter=0.12, point_size=8, alpha=0.35, showfliers=False):
    groups = []
    labels = []

    for m in mice:
        vals = data.loc[data[mouse_col] == m, col].dropna().values
        groups.append(vals)
        labels.append(str(m))

    ax.boxplot(
    groups,
    showfliers=showfliers,
    medianprops=dict(color="black", linewidth=1),
)

    # overlay points (jittered)
    for i, vals in enumerate(groups, start=1):
        if len(vals) == 0:
            continue
        x = i + rng.uniform(-jitter, jitter, size=len(vals))
        ax.scatter(x, vals, s=point_size, alpha=alpha)

    ax.set_xticks(range(1, len(labels) + 1))
    ax.set_xticklabels(labels, rotation=90)
    ax.set_xlabel("MouseID (mouse_name)")
    ax.set_ylabel(col)
    ax.set_title(f"{col} by MouseID (boxplot + points)")

# Example plot for spatial_info
plt.figure()
ax = plt.gca()
boxplot_with_points(ax, df_linear_track, "spatial_info", mice)
plt.tight_layout()
plt.show()

# Multi-page PDF (save next to the Excel file)
pdf_path = os.path.join(
    linear_track_DIR,
    "linear_track_all_numeric_boxplots_with_points_by_mouse.pdf"
)

with PdfPages(pdf_path) as pdf:
    fig = plt.figure(figsize=(11.69, 8.27))
    txt = (
        "Open field dataset\n"
        f"Rows: {len(df_linear_track):,} | Columns: {df_linear_track.shape[1]:,}\n"
        f"MouseID column: {mouse_col}\n"
        "Contents: one boxplot per numeric column, grouped by MouseID,\n"
        "with individual observations overlaid (jittered points).\n"
    )
    fig.text(0.05, 0.9, txt, fontsize=16, va="top")
    plt.axis("off")
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)

    for col in numeric_cols:
        fig = plt.figure(figsize=(11.69, 8.27))
        ax = fig.add_subplot(111)
        boxplot_with_points(ax, df_linear_track, col, mice, jitter=0.12, point_size=6, alpha=0.4, showfliers=False)
        fig.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

pdf_path

### Generate violin plots with overlaid points for all numeric columns, grouped by mouse.
#### Create a multi-page PDF with one plot per numeric column showing distributions across mice.

In [ ]:
"""Violin plots with overlaid points (one colour per mouse, using tab20b)
- Mean and median lines styled differently from default (e.g. mean in green, median in black)
- Legend for mean/median line styles (not from tab20/tab20b)
- Points use tab20b, one colour per mouse
"""

mouse_col = "mouse_name"
mice = sorted(df_linear_track[mouse_col].dropna().unique().tolist())
numeric_cols = [c for c in df_linear_track.select_dtypes(include=[np.number]).columns if c != mouse_col]

rng = np.random.default_rng(0)
point_cmap = plt.get_cmap("tab20b")  # <- use tab20b everywhere for points

# Mean/median legend (NOT from tab20/tab20b)
legend_elements = [
    Line2D([0], [0], color="#009E73", lw=2, linestyle="-", label="Mean"),
    Line2D([0], [0], color="black", lw=2, linestyle="-", label="Median"),
]

def violin_with_points(ax, data, col, mice, jitter=0.12, point_size=4, alpha=0.3):
    groups = []
    labels = []
    for m in mice:
        vals = data.loc[data[mouse_col] == m, col].dropna().values
        groups.append(vals)
        labels.append(str(m))

    parts = ax.violinplot(
        groups,
        showmeans=True,
        showmedians=True,
        showextrema=False,
    )

    # Mean / median styling
    if parts.get("cmeans") is not None:
        parts["cmeans"].set_color("#009E73")
        parts["cmeans"].set_linewidth(2)
        parts["cmeans"].set_linestyle("-")

    if parts.get("cmedians") is not None:
        parts["cmedians"].set_color("black")
        parts["cmedians"].set_linewidth(2)
        parts["cmedians"].set_linestyle("-")

    # Overlay points (jittered) — tab20b, one colour per mouse
    for i, vals in enumerate(groups, start=1):
        if len(vals) == 0:
            continue
        x = i + rng.uniform(-jitter, jitter, size=len(vals))
        ax.scatter(
            x,
            vals,
            s=point_size,
            alpha=alpha,
            color=point_cmap((i - 1) % 20),
        )

    ax.set_xticks(range(1, len(labels) + 1))
    ax.set_xticklabels(labels, rotation=90)
    ax.set_xlabel("MouseID (mouse_name)")
    ax.set_ylabel(col)
    ax.set_title(f"{col} by MouseID (violin + points)")
    ax.legend(handles=legend_elements, loc="upper right")


# Example plot
plt.figure()
ax = plt.gca()
violin_with_points(ax, df_linear_track, "spatial_info", mice)
plt.tight_layout()
plt.show()

# Multi-page PDF (PDF uses the function; no extra legend call needed)
savepdf_path = os.path.join(
    linear_track_DIR,
    "linear_track_all_numeric_violinplots_with_points_by_mouse.pdf",
)
print(f"PDF saved to: {savepdf_path}")

with PdfPages(savepdf_path) as pdf:
    fig = plt.figure(figsize=(11.69, 8.27))
    txt = (
        "Open field dataset\n"
        f"Rows: {len(df_linear_track):,} | Columns: {df_linear_track.shape[1]:,}\n"
        f"MouseID column: {mouse_col}\n"
        "Contents: one violin plot per numeric column, grouped by MouseID,\n"
        "with individual observations overlaid (jittered points).\n"
        "Points use tab20b.\n"
    )
    fig.text(0.05, 0.9, txt, fontsize=16, va="top")
    plt.axis("off")
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)

    for col in numeric_cols:
        fig = plt.figure(figsize=(11.69, 8.27))
        ax = fig.add_subplot(111)
        violin_with_points(ax, df_linear_track, col, mice)
        fig.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

savepdf_path

**Description of the next cell**

This cell builds a PDF report of distribution diagnostics for numeric columns in `df_linear_track`. For each target column (currents/densities, or all numeric if none match), it:

- Plots a histogram of absolute values with the y axis as percentage of observations.
- Draws a custom summary “box” above the histogram:
    - whiskers = min/max
    - box = 16th–84th percentiles
    - median line
    - filled circle = geometric mean
    - open circle = arithmetic mean
- Fits log‑normal (solid red), gamma (dashed blue), and normal (dotted purple) PDFs.
- Tests each fit with a two‑sample Anderson–Darling test against synthetic samples from the fitted distribution.
    - If none pass (p > 0.05), it overlays an Akima spline as a nonparametric fit.
- Annotates each plot with Shapiro–Wilk p‑value, skewness, and AD p‑values.
- Saves all plots to `linear_track_histograms_abs_with_fits_and_custom_boxplots.pdf` in `linear_track_DIR`.

**Tests performed**
- **Shapiro–Wilk test** for normality of the absolute values (reported if 3–5000 samples).
- **Anderson–Darling two‑sample test** comparing data vs simulated samples from each fitted distribution (log‑normal, gamma, normal). A fit is considered acceptable when p > 0.05.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

# Requires SciPy
from scipy import stats
from scipy.interpolate import Akima1DInterpolator

def _geometric_mean_positive(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    x = x[x > 0]
    if len(x) == 0:
        return np.nan
    return float(np.exp(np.mean(np.log(x))))

def _custom_box_on_axis(ax, x, y=0.5):
    """
    Draws the requested "boxplot" on a blank axis with the SAME x-scale as the histogram.
    whiskers: min/max
    hinges: p16/p84
    median: p50
    filled circle: geometric mean
    empty circle: arithmetic mean
    """
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        ax.set_axis_off()
        return

    xmin, xmax = float(np.min(x)), float(np.max(x))
    p16, p50, p84 = np.percentile(x, [16, 50, 84])
    gmean = _geometric_mean_positive(x)
    amean = float(np.mean(x))

    # baseline
    ax.set_ylim(0, 1)
    ax.set_yticks([])
    ax.spines["left"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)

    # whisker line + caps
    ax.hlines(y, xmin, xmax, color="black", linewidth=1.5)
    ax.vlines([xmin, xmax], y - 0.08, y + 0.08, color="black", linewidth=1.5)

    # box from p16 to p84
    ax.add_patch(
        plt.Rectangle(
            (p16, y - 0.18),
            p84 - p16,
            0.36,
            fill=False,
            edgecolor="black",
            linewidth=1.5,
        )
    )

    # median line
    ax.vlines(p50, y - 0.18, y + 0.18, color="black", linewidth=2)

    # filled circle = geometric mean
    if np.isfinite(gmean):
        ax.plot(gmean, y, marker="o", markersize=6, color="black")

    # empty circle = arithmetic mean
    ax.plot(amean, y, marker="o", markersize=6, markerfacecolor="white", markeredgecolor="black")

def _hist_fraction(ax, x, bins="fd"):
    """
    Histogram where y-axis is fraction of observations (sums to 1).
    Returns bin_edges, heights.
    """
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return None, None

    # If bins is an estimator string (e.g. "fd", "auto"), compute bin edges first.
    if isinstance(bins, str):
        bin_edges = np.histogram_bin_edges(x, bins=bins)
        bins_to_use = bin_edges
    else:
        bins_to_use = bins

    weights = np.ones_like(x) * (100 / len(x))  # weights to get percentage (sum to 100)

    heights, bin_edges, _ = ax.hist(
        x,
        bins=bins_to_use,
        weights=weights,
        edgecolor="black",
        linewidth=1.0,
        facecolor="white",
        hatch="///",
    )
    return bin_edges, heights

def _scale_to_hist_max(y, hist_max):
    y = np.asarray(y, dtype=float)
    ymax = np.nanmax(y) if np.any(np.isfinite(y)) else np.nan
    if not np.isfinite(ymax) or ymax <= 0 or hist_max <= 0:
        return y
    return y * (hist_max / ymax)

def _ad_fit_ok(data, dist_rvs, alpha=0.05, n_synth=5000, rng=None):
    """
    “Anderson–Darling test did not find difference significant” implemented as:
    two-sample AD test between data and synthetic samples from the fitted distribution.
    Accept fit if p-value (significance_level/100) > alpha.
    """
    if rng is None:
        rng = np.random.default_rng(0)

    data = np.asarray(data, dtype=float)
    data = data[np.isfinite(data)]
    if len(data) < 8:
        return False, np.nan  # too small for stable testing

    synth = dist_rvs(size=n_synth, random_state=rng)
    synth = np.asarray(synth, dtype=float)
    synth = synth[np.isfinite(synth)]
    if len(synth) < 8:
        return False, np.nan

    res = stats.anderson_ksamp([data, synth])
    # res.significance_level is in percent (approx)
    p = float(res.significance_level) / 100.0
    return (p > alpha), p

def plot_hist_with_fits_and_box(
    data,
    col,
    *,
    bins="fd",
    alpha_ad=0.05,
    rng=None,
):
    if rng is None:
        rng = np.random.default_rng(0)

    x_abs = np.abs(np.asarray(data[col].dropna().values, dtype=float))
    x_abs = x_abs[np.isfinite(x_abs)]

    # For fitting lognorm/gamma we need > 0
    x_fit = x_abs[x_abs > 0]

    # Stats shown on plot
    skew = float(stats.skew(x_abs, nan_policy="omit")) if len(x_abs) else np.nan
    shapiro_p = np.nan
    if 3 <= len(x_abs) <= 5000:
        shapiro_p = float(stats.shapiro(x_abs).pvalue)

    # Figure layout: boxplot on top, histogram below (shared x)
    fig = plt.figure(figsize=(11.69, 8.27))
    gs = fig.add_gridspec(nrows=2, ncols=1, height_ratios=[1, 6], hspace=0.05)
    ax_box = fig.add_subplot(gs[0, 0])
    ax = fig.add_subplot(gs[1, 0], sharex=ax_box)

    # Draw custom box “plot”
    _custom_box_on_axis(ax_box, x_abs)
    ax_box.tick_params(axis="x", labelbottom=False)

    # Histogram
    bin_edges, heights = _hist_fraction(ax, x_abs, bins=bins)
    if bin_edges is None:
        ax.set_title(f"{col} (no data)")
        return fig

    hist_max = float(np.max(heights)) if len(heights) else 0.0

    # X grid for PDFs
    xmin, xmax = float(bin_edges[0]), float(bin_edges[-1])
    xx = np.linspace(xmin, xmax, 400)

    # Initialize flags BEFORE use
    lognorm_ok, gamma_ok, normal_ok = False, False, False
    lognorm_p, gamma_p, normal_p = np.nan, np.nan, np.nan

    # Try log-normal (solid red)
    if len(x_fit) >= 8:
        s, loc, scale = stats.lognorm.fit(x_fit, floc=0)
        yy = stats.lognorm.pdf(xx, s, loc=loc, scale=scale)
        yy_scaled = _scale_to_hist_max(yy, hist_max)
        lognorm_ok, lognorm_p = _ad_fit_ok(
            x_fit,
            lambda size, random_state=None: stats.lognorm.rvs(
                s, loc=loc, scale=scale, size=size, random_state=random_state
            ),
            alpha=alpha_ad,
            rng=rng,
        )
        ax.plot(xx, yy_scaled, color="red", linewidth=2.5)

    # Try gamma (dashed blue)
    if len(x_fit) >= 8:
        a, loc, scale = stats.gamma.fit(x_fit, floc=0)
        yy = stats.gamma.pdf(xx, a, loc=loc, scale=scale)
        yy_scaled = _scale_to_hist_max(yy, hist_max)
        gamma_ok, gamma_p = _ad_fit_ok(
            x_fit,
            lambda size, random_state=None: stats.gamma.rvs(
                a, loc=loc, scale=scale, size=size, random_state=random_state
            ),
            alpha=alpha_ad,
            rng=rng,
        )
        ax.plot(xx, yy_scaled, color="blue", linestyle="--", linewidth=2.5)

    # Try normal (dotted purple)
    if len(x_abs) >= 8:
        mu, sigma = stats.norm.fit(x_abs)
        if np.isfinite(sigma) and sigma > 0:
            yy = stats.norm.pdf(xx, loc=mu, scale=sigma)
            yy_scaled = _scale_to_hist_max(yy, hist_max)
            normal_ok, normal_p = _ad_fit_ok(
                x_abs,
                lambda size, random_state=None: stats.norm.rvs(
                    loc=mu, scale=sigma, size=size, random_state=random_state
                ),
                alpha=alpha_ad,
                rng=rng,
            )
            ax.plot(xx, yy_scaled, color="#7B3294", linestyle=":", linewidth=2.5)

    fit_ok = bool(lognorm_ok or gamma_ok or normal_ok)

    # If none acceptable -> Akima spline (grey)
    if not fit_ok:
        centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
        ak = Akima1DInterpolator(centers, heights)
        yy = ak(xx)
        yy = np.clip(yy, 0, None)
        ax.plot(xx, yy, color="grey", linewidth=2.0)

    # Labels / annotation
    ax.set_ylabel("Percent of cells (per bin)")
    ax.set_xlabel(f"|{col}| (absolute values)")
    ax.set_title(col)

    info = [
        f"Shapiro p={shapiro_p:.3g}" if np.isfinite(shapiro_p) else "Shapiro p=NA",
        f"skew={skew:.3g}" if np.isfinite(skew) else "skew=NA",
        f"AD p(lognorm)={lognorm_p:.3g}" if np.isfinite(lognorm_p) else "AD p(lognorm)=NA",
        f"AD p(gamma)={gamma_p:.3g}" if np.isfinite(gamma_p) else "AD p(gamma)=NA",
        f"AD p(normal)={normal_p:.3g}" if np.isfinite(normal_p) else "AD p(normal)=NA",
    ]
    ax.text(
        0.98, 0.98, "\n".join(info),
        transform=ax.transAxes, ha="right", va="top", fontsize=10
    )

    return fig


# ---- Run for all relevant columns and save a PDF next to your Excel file ----

# Prefer columns that look like currents/densities; fall back to all numeric
numeric_cols = df_linear_track.select_dtypes(include=[np.number]).columns.tolist()
target_cols = [c for c in numeric_cols if ("current" in c.lower() or "density" in c.lower())]
if len(target_cols) == 0:
    target_cols = numeric_cols

pdf_path = os.path.join(linear_track_DIR, "linear_track_histograms_abs_with_fits_and_custom_boxplots.pdf")
rng = np.random.default_rng(0)

with PdfPages(pdf_path) as pdf:
    # --- First page: description / legend page ---
    fig = plt.figure(figsize=(11.69, 8.27))
    description_lines = [
        "Histogram diagnostics report (absolute values)",
        "",
        "Each PDF page (for one column col) has two stacked panels:",
        "",
        "Top panel (custom “boxplot”) on the same x-scale as the histogram:",
        "  - whiskers = min → max of |col|",
        "  - box = 16th → 84th percentile",
        "  - vertical line = median (50th)",
        "  - filled circle = geometric mean (computed on positive values only)",
        "  - open circle = arithmetic mean",
        "",
        "Bottom panel (histogram of absolute values):",
        "  - x-axis: |col| (absolute values of that metric)",
        "  - bars: percent of cells in each bin (weights sum to 100)",
        "  - overlaid fit lines (scaled to histogram peak for visual comparison):",
        "      - log-normal = solid red",
        "      - gamma = dashed blue",
        "      - normal = dotted purple",
        "      - if none pass AD test: Akima spline = grey",
        "  - text annotation: Shapiro–Wilk p-value, skewness, and AD p-values",
    ]

    txt = "\n".join(textwrap.fill(line, width=105) for line in description_lines)
    fig.text(0.05, 0.95, txt, fontsize=14, va="top", family="monospace")
    plt.axis("off")
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)

    # --- Then the plots ---
    for col in target_cols:
        fig = plot_hist_with_fits_and_box(df_linear_track, col, bins="fd", alpha_ad=0.05, rng=rng)
        fig.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

fig = plot_hist_with_fits_and_box(df_linear_track, col, bins="fd", alpha_ad=0.05, rng=rng)

print(f"PDF with histograms, fits, and custom boxplots saved to: {pdf_path}")

In [ ]:
"""2-way ANOVA to test for effects of Experimenter and Age_weeks on numeric columns
- Use OLS from statsmodels to fit the model: dependent variable ~ C(Experimenter) + Age_weeks + C(Experimenter):Age_weeks
- Extract p-values for Experimenter, Age_weeks, and their interaction
- Create a summary table with columns: column, n_observations, p_Experimenter, p_Age_weeks, p_Interaction, sig_Experimenter, sig_Age_weeks, sig_Interaction
- Significance stars: '***' for p < 0.001, '**' for p < 0.01, '*' for p < 0.05, 'ns' otherwise  
"""

from scipy import stats
import warnings
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

warnings.filterwarnings('ignore')

# Prepare data for 2-way ANOVA
# We'll use experimenter and a categorical age variable (since age is continuous, we can use it directly or bin it)
# For 2-way ANOVA with continuous age and categorical experimenter, we'll use OLS from statsmodels


# Create results storage
anova_results = []

# For each numeric column, run 2-way ANOVA with Experimenter and Age_weeks as factors
for col in numeric_cols:
    if col in ['tetrode', 'cluster', 'mouse_name']:
        continue  # Skip ID-type columns
    
    # Prepare data without missing values
    test_data = df_linear_track[[col, 'Experimenter', 'Age_weeks']].dropna()
    
    if len(test_data) < 10:  # Skip if too few observations
        continue
    
    try:
        # Fit 2-way ANOVA model: dependent variable ~ factor1 + factor2 + interaction
        formula = f'Q("{col}") ~ C(Experimenter) + Age_weeks + C(Experimenter):Age_weeks'
        model = ols(formula, data=test_data).fit()
        anova_table = anova_lm(model, typ=2)
        
        # Extract p-values
        p_experimenter = anova_table.loc['C(Experimenter)', 'PR(>F)']
        p_age = anova_table.loc['Age_weeks', 'PR(>F)']
        p_interaction = anova_table.loc['C(Experimenter):Age_weeks', 'PR(>F)']
        
        anova_results.append({
            'column': col,
            'n_observations': len(test_data),
            'p_Experimenter': p_experimenter,
            'p_Age_weeks': p_age,
            'p_Interaction': p_interaction,
            'sig_Experimenter': '***' if p_experimenter < 0.001 else '**' if p_experimenter < 0.01 else '*' if p_experimenter < 0.05 else 'ns',
            'sig_Age_weeks': '***' if p_age < 0.001 else '**' if p_age < 0.01 else '*' if p_age < 0.05 else 'ns',
            'sig_Interaction': '***' if p_interaction < 0.001 else '**' if p_interaction < 0.01 else '*' if p_interaction < 0.05 else 'ns',
        })
    except Exception as e:
        print(f"Could not run ANOVA for {col}: {e}")
        continue

# Create results dataframe
df_anova_results = pd.DataFrame(anova_results)

# Display results
display_df("2-way ANOVA Results: Effect of Experimenter and Age_weeks on numeric columns", df_anova_results)

# Save results
anova_save_path = os.path.join(linear_track_DIR, "linear_track_2way_anova_experimenter_age.csv")
df_anova_results.to_csv(anova_save_path, index=False)
print(f"\nANOVA results saved to: {anova_save_path}")

# Summary of significant effects
print("\n=== SUMMARY OF SIGNIFICANT EFFECTS (p < 0.05) ===")
print(f"\nExperimenter effects: {(df_anova_results['sig_Experimenter'] != 'ns').sum()} out of {len(df_anova_results)} columns")
print(f"Age_weeks effects: {(df_anova_results['sig_Age_weeks'] != 'ns').sum()} out of {len(df_anova_results)} columns")
print(f"Interaction effects: {(df_anova_results['sig_Interaction'] != 'ns').sum()} out of {len(df_anova_results)} columns")

In [ ]:
"""Visualization of ANOVA results
Create a heatmap of p-values
Plot 1: Heatmap of p-values for Experimenter, Age_weeks, and Interaction
Plot 2: Bar plot of -log10(p-values) for Experimenter
Plot 3: Bar plot of -log10(p-values) for Age_weeks
"""


import seaborn as sns

# Visualization of ANOVA results

# 1. Create a heatmap of p-values
fig, axes = plt.subplots(3, 1, figsize=(22, 15))

# Prepare data for heatmap
heatmap_data = df_anova_results[['column', 'p_Experimenter', 'p_Age_weeks', 'p_Interaction']].set_index('column')

# Plot 1: Heatmap of p-values
sns.heatmap(heatmap_data.T, annot=True, fmt='.4f', cmap='RdYlGn_r', 
            vmin=0, vmax=0.1, ax=axes[0], cbar_kws={'label': 'p-value'})
axes[0].set_title('ANOVA p-values: Effect of Experimenter, Age_weeks, and Interaction')
axes[0].set_ylabel('Factor')
axes[0].set_xlabel('')

# Plot 2: Bar plot of -log10(p-values) for Experimenter
neg_log_p_exp = -np.log10(df_anova_results['p_Experimenter'])
axes[1].bar(range(len(df_anova_results)), neg_log_p_exp, color='steelblue')
axes[1].axhline(y=-np.log10(0.05), color='red', linestyle='--', label='p=0.05')
axes[1].axhline(y=-np.log10(0.01), color='orange', linestyle='--', label='p=0.01')
axes[1].axhline(y=-np.log10(0.001), color='darkred', linestyle='--', label='p=0.001')
axes[1].set_xticks(range(len(df_anova_results)))
axes[1].set_xticklabels(df_anova_results['column'], rotation=90, ha='right')
axes[1].set_ylabel('-log10(p-value)')
axes[1].set_title('Experimenter Effect Significance')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

# Plot 3: Bar plot of -log10(p-values) for Age_weeks
neg_log_p_age = -np.log10(df_anova_results['p_Age_weeks'])
axes[2].bar(range(len(df_anova_results)), neg_log_p_age, color='darkorange')
axes[2].axhline(y=-np.log10(0.05), color='red', linestyle='--', label='p=0.05')
axes[2].axhline(y=-np.log10(0.01), color='orange', linestyle='--', label='p=0.01')
axes[2].axhline(y=-np.log10(0.001), color='darkred', linestyle='--', label='p=0.001')
axes[2].set_xticks(range(len(df_anova_results)))
axes[2].set_xticklabels(df_anova_results['column'], rotation=90, ha='right')
axes[2].set_ylabel('-log10(p-value)')
axes[2].set_xlabel('Metric')
axes[2].set_title('Age_weeks Effect Significance')
axes[2].legend()
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Save the ANOVA visualization
anova_plot_path = os.path.join(linear_track_DIR, "linear_track_anova_results_visualization.pdf")
fig.savefig(anova_plot_path, bbox_inches='tight')
print(f"\nANOVA visualization saved to: {anova_plot_path}")

In [ ]:
""" Visualization of t-test results from experimenter comparison
Plot 1: Bar plot of -log10(p-values) for t-test
Plot 2: Bar plot of -log10(p-values) for Mann–Whitney U test
"""

from scipy.stats import ttest_ind, mannwhitneyu

def _sig_stars(p):
    if p is None or not np.isfinite(p):
        return "NA"
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"

# Create boxplots grouped by Experimenter for all numeric columns
experimenters = sorted(df_linear_track["Experimenter"].dropna().unique().tolist())

pdf_experimenter_path = os.path.join(
    linear_track_DIR,
    "linear_track_boxplots_by_experimenter.pdf"
)

experimenter_stats = []

with PdfPages(pdf_experimenter_path) as pdf:
    # Title page
    fig = plt.figure(figsize=(11.69, 8.27))
    txt = (
        "Open field dataset - Analysis by Experimenter\n"
        f"Rows: {len(df_linear_track):,} | Columns: {df_linear_track.shape[1]:,}\n"
        f"Experimenters: {', '.join(map(str, experimenters))}\n"
        "Contents: one boxplot per numeric column, grouped by Experimenter,\n"
        "with individual observations overlaid (jittered points).\n"
        "Each page is annotated with Mann–Whitney U (and t-test) results when 2 groups are present.\n"
    )
    fig.text(0.05, 0.9, txt, fontsize=16, va="top")
    plt.axis("off")
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)

    cols_to_plot = [c for c in numeric_cols if c not in ["tetrode", "cluster", "mouse_name", "Age_weeks"]]

    for col in cols_to_plot:
        fig = plt.figure(figsize=(11.69, 8.27))
        ax = fig.add_subplot(111)

        # Prepare data per experimenter
        groups = []
        ns = []
        for exp in experimenters:
            vals = df_linear_track.loc[df_linear_track["Experimenter"] == exp, col].dropna().values.astype(float)
            groups.append(vals)
            ns.append(len(vals))

        # Per-group summaries (always shown)
        group_summary_lines = []
        for exp, vals in zip(experimenters, groups):
            if len(vals) == 0:
                group_summary_lines.append(f"{exp}: n=0, mean=NA, median=NA")
            else:
                group_summary_lines.append(
                    f"{exp}: n={len(vals)}, mean={np.mean(vals):.3g}, median={np.median(vals):.3g}"
                )

        # Boxplot + points
        ax.boxplot(groups, showfliers=False, medianprops=dict(color="black", linewidth=1))
        for i, vals in enumerate(groups, start=1):
            if len(vals) == 0:
                continue
            x = i + rng.uniform(-0.12, 0.12, size=len(vals))
            ax.scatter(x, vals, s=6, alpha=0.4)

        ax.set_xticks(range(1, len(experimenters) + 1))
        ax.set_xticklabels([str(e) for e in experimenters], rotation=0)
        ax.set_xlabel("Experimenter")
        ax.set_ylabel(col)
        ax.set_title(f"{col} by Experimenter")

        # --- Stats + annotation (only meaningful when exactly 2 experimenters) ---
        t_pval = np.nan
        u_stat = np.nan
        u_pval = np.nan
        stat_note = ""

        if len(experimenters) == 2:
            g1, g2 = groups[0], groups[1]
            if (len(g1) >= 3) and (len(g2) >= 3):
                # t-test
                try:
                    _, t_pval = ttest_ind(g1, g2, nan_policy="omit")
                except Exception:
                    t_pval = np.nan

                # Mann–Whitney U
                try:
                    u_stat, u_pval = mannwhitneyu(g1, g2, alternative="two-sided")
                except Exception:
                    u_stat, u_pval = np.nan, np.nan

                stat_note = (
                    f"t-test p={t_pval:.3g} ({_sig_stars(t_pval)})\n"
                    f"Mann–Whitney U={u_stat:.3g}, p={u_pval:.3g} ({_sig_stars(u_pval)})\n"
                    + "\n".join(group_summary_lines)
                )
            else:
                stat_note = f"Stats not run (need n>=3 per group)\n" f"n={ns[0]} vs {ns[1]}"
        else:
            stat_note = f"Mann–Whitney shown only for 2 groups\n(groups found: {len(experimenters)})"

        ax.text(
            0.98, 0.98, stat_note,
            transform=ax.transAxes,
            ha="right", va="top",
            fontsize=11,
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="black", alpha=0.8),
        )

        # Save stats row (only if 2 groups and enough data)
        if len(experimenters) == 2 and (ns[0] >= 3) and (ns[1] >= 3):
            exp1, exp2 = experimenters[0], experimenters[1]
            experimenter_stats.append({
                "column": col,
                f"n_{exp1}": ns[0],
                f"n_{exp2}": ns[1],
                f"mean_{exp1}": float(np.mean(groups[0])) if ns[0] else np.nan,
                f"mean_{exp2}": float(np.mean(groups[1])) if ns[1] else np.nan,
                f"median_{exp1}": float(np.median(groups[0])) if ns[0] else np.nan,
                f"median_{exp2}": float(np.median(groups[1])) if ns[1] else np.nan,
                "t_pval": float(t_pval) if np.isfinite(t_pval) else np.nan,
                "t_sig": _sig_stars(t_pval),
                "mann_whitney_u": float(u_stat) if np.isfinite(u_stat) else np.nan,
                "mann_whitney_pval": float(u_pval) if np.isfinite(u_pval) else np.nan,
                "mw_sig": _sig_stars(u_pval),
            })

        fig.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

print(f"Experimenter boxplots saved to: {pdf_experimenter_path}")

# Save stats table (if any)
df_experimenter_stats = pd.DataFrame(experimenter_stats)
display_df("Statistical Tests: Differences between Experimenters", df_experimenter_stats)

experimenter_stats_path = os.path.join(linear_track_DIR, "linear_track_experimenter_statistics.csv")
df_experimenter_stats.to_csv(experimenter_stats_path, index=False)
print(f"\nExperimenter statistics saved to: {experimenter_stats_path}")

In [ ]:
# Visualization of t-test results from experimenter comparison

fig, axes = plt.subplots(2, 2, figsize=(15, 12) )

# Plot 1: Bar plot of -log10(p-values) for t-test
ax = axes[0, 0]
neg_log_p_ttest = -np.log10(df_experimenter_stats['t_pval'])
bars = ax.bar(range(len(df_experimenter_stats)), neg_log_p_ttest, color='steelblue')
ax.axhline(y=-np.log10(0.05), color='red', linestyle='--', label='p=0.05', linewidth=2)
ax.axhline(y=-np.log10(0.01), color='orange', linestyle='--', label='p=0.01', linewidth=2)
ax.axhline(y=-np.log10(0.001), color='darkred', linestyle='--', label='p=0.001', linewidth=2)
ax.set_xticks(range(len(df_experimenter_stats)))
ax.set_xticklabels(df_experimenter_stats['column'], rotation=90, ha='right')
ax.set_ylabel('-log10(p-value)')
ax.set_title('T-test: Differences between Experimenters')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Plot 2: Bar plot of -log10(p-values) for Mann-Whitney U test
ax = axes[0, 1]
neg_log_p_mw = -np.log10(df_experimenter_stats['mann_whitney_pval'])
bars = ax.bar(range(len(df_experimenter_stats)), neg_log_p_mw, color='darkorange')
ax.axhline(y=-np.log10(0.05), color='red', linestyle='--', label='p=0.05', linewidth=2)
ax.axhline(y=-np.log10(0.01), color='orange', linestyle='--', label='p=0.01', linewidth=2)
ax.axhline(y=-np.log10(0.001), color='darkred', linestyle='--', label='p=0.001', linewidth=2)
ax.set_xticks(range(len(df_experimenter_stats)))
ax.set_xticklabels(df_experimenter_stats['column'], rotation=90, ha='right')
ax.set_ylabel('-log10(p-value)')
ax.set_title('Mann-Whitney U test: Differences between Experimenters')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Plot 3: Comparison of means between experimenters
ax = axes[1, 0]
x_pos = np.arange(len(df_experimenter_stats))
width = 0.35
exp1, exp2 = experimenters[0], experimenters[1]
bars1 = ax.bar(x_pos - width/2, df_experimenter_stats[f'mean_{exp1}'], width, label=exp1, alpha=0.8)
bars2 = ax.bar(x_pos + width/2, df_experimenter_stats[f'mean_{exp2}'], width, label=exp2, alpha=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels(df_experimenter_stats['column'], rotation=90, ha='right')
ax.set_ylabel('Mean value')
ax.set_title('Mean values by Experimenter')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Plot 4: Heatmap of significance
ax = axes[1, 1]

# be robust to either naming convention
t_sig_col = "sig_ttest" if "sig_ttest" in df_experimenter_stats.columns else "t_sig"
mw_sig_col = "sig_mannwhitney" if "sig_mannwhitney" in df_experimenter_stats.columns else "mw_sig"

sig_data = df_experimenter_stats[["column", t_sig_col, mw_sig_col]].set_index("column")
sig_data = sig_data.rename(columns={t_sig_col: "t-test", mw_sig_col: "Mann–Whitney"})

sig_numeric = sig_data.replace({"***": 3, "**": 2, "*": 1, "ns": 0, "NA": np.nan})

sns.heatmap(
    sig_numeric.T,
    annot=sig_data.T.values,
    fmt="",
    cmap="RdYlGn",
    vmin=0,
    vmax=3,
    ax=ax,
    cbar_kws={"label": "Significance level"},
)
ax.set_title("Significance levels: Experimenter comparison")
ax.set_ylabel("Test")

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
import re


import warnings
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

warnings.filterwarnings("ignore")


EXPERIMENTER_COL = "Experimenter"

def _find_col(df, names):
    def norm(s):
        return re.sub(r"[\s_]+", "", str(s)).lower()
    norm_map = {norm(c): c for c in df.columns}
    for n in names:
        key = norm(n)
        if key in norm_map:
            return norm_map[key]
    raise KeyError(f"Could not find any of {names}. Available columns: {list(df.columns)}")

CELLTYPE_COL = _find_col(df_linear_track, ["Cell Type", "Cell type", "cell_type", "celltype"])
print("Using CELLTYPE_COL =", CELLTYPE_COL)

def _sig_stars(p):
    if p is None or not np.isfinite(p):
        return "NA"
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"

# numeric metrics to analyze/plot
numeric_cols = df_linear_track.select_dtypes(include=[np.number]).columns.tolist()
cols_to_use = [c for c in numeric_cols if c not in ["tetrode", "cluster", "mouse_name"]]

# ---------- 1) 2-way ANOVA: Experimenter x Cell type ----------
anova_rows = []
term_exp = f'C(Q("{EXPERIMENTER_COL}"))'
term_ct  = f'C(Q("{CELLTYPE_COL}"))'
term_int = f'{term_exp}:{term_ct}'

for col in cols_to_use:
    test_data = df_linear_track[[col, EXPERIMENTER_COL, CELLTYPE_COL]].dropna()
    if len(test_data) < 10:
        continue
    if test_data[EXPERIMENTER_COL].nunique() < 2 or test_data[CELLTYPE_COL].nunique() < 2:
        continue

    try:
        formula = f'Q("{col}") ~ {term_exp} * {term_ct}'
        model = ols(formula, data=test_data).fit()
        aov = anova_lm(model, typ=2)

        p_exp = float(aov.loc[term_exp, "PR(>F)"]) if term_exp in aov.index else np.nan
        p_ct  = float(aov.loc[term_ct,  "PR(>F)"]) if term_ct  in aov.index else np.nan
        p_int = float(aov.loc[term_int, "PR(>F)"]) if term_int in aov.index else np.nan

        anova_rows.append({
            "column": col,
            "n_observations": int(len(test_data)),
            "n_experimenters": int(test_data[EXPERIMENTER_COL].nunique()),
            "n_cell_types": int(test_data[CELLTYPE_COL].nunique()),
            "p_Experimenter": p_exp,
            "p_CellType": p_ct,
            "p_Interaction": p_int,
            "sig_Experimenter": _sig_stars(p_exp),
            "sig_CellType": _sig_stars(p_ct),
            "sig_Interaction": _sig_stars(p_int),
        })
    except Exception as e:
        print(f"Could not run ANOVA for {col}: {e}")

df_anova_exp_celltype = pd.DataFrame(anova_rows).sort_values("p_Experimenter", na_position="last")
display_df("2-way ANOVA: Experimenter x Cell type", df_anova_exp_celltype)

anova_save_path = os.path.join(linear_track_DIR, "linear_track_2way_anova_experimenter_celltype.csv")
df_anova_exp_celltype.to_csv(anova_save_path, index=False)
print(f"Saved ANOVA table to: {anova_save_path}")

# ---------- 2) Boxplots PDF: metric by Cell type (hue=Experimenter) ----------
pdf_path = os.path.join(linear_track_DIR, "linear_track_boxplots_by_experimenter_and_celltype.pdf")

with PdfPages(pdf_path) as pdf:
    # title page
    fig = plt.figure(figsize=(11.69, 8.27))
    exp_levels = sorted(df_linear_track[EXPERIMENTER_COL].dropna().unique().tolist())
    ct_levels = sorted(df_linear_track[CELLTYPE_COL].dropna().unique().tolist())
    txt = (
        "Open field dataset - Boxplots by Experimenter and Cell type\n"
        f"Experimenters: {', '.join(map(str, exp_levels))}\n"
        f"Cell types: {', '.join(map(str, ct_levels))}\n"
        "Each page: boxplot (no fliers) + jittered points\n"
        "x = Cell type, hue = Experimenter\n"
    )
    fig.text(0.05, 0.9, txt, fontsize=14, va="top")
    plt.axis("off")
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)

    for col in cols_to_use:
        plot_data = df_linear_track[[col, EXPERIMENTER_COL, CELLTYPE_COL]].dropna()
        if len(plot_data) == 0:
            continue
        if plot_data[CELLTYPE_COL].nunique() < 2:
            continue

        fig = plt.figure(figsize=(11.69, 8.27))
        ax = fig.add_subplot(111)

        sns.boxplot(
            data=plot_data,
            x=CELLTYPE_COL,
            y=col,
            hue=EXPERIMENTER_COL,
            showfliers=False,
            ax=ax,
        )
        sns.stripplot(
            data=plot_data,
            x=CELLTYPE_COL,
            y=col,
            hue=EXPERIMENTER_COL,
            dodge=True,
            jitter=0.2,
            size=2.5,
            alpha=0.35,
            linewidth=0,
            ax=ax,
        )

        # de-duplicate legend (boxplot + stripplot both add)
        handles, labels = ax.get_legend_handles_labels()
        by_label = dict(zip(labels, handles))
        ax.legend(
            by_label.values(),
            by_label.keys(),
            title=EXPERIMENTER_COL,
            bbox_to_anchor=(1.02, 1),
            loc="upper left",
            borderaxespad=0,
        )

        ax.set_title(f"{col} by Cell type (hue=Experimenter)")
        ax.set_xlabel(CELLTYPE_COL)
        ax.set_ylabel(col)
        ax.tick_params(axis="x", rotation=45)

        fig.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

print(f"Saved boxplots PDF to: {pdf_path}")

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
import re

import warnings
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

warnings.filterwarnings("ignore")

EXPERIMENTER_COL = "Experimenter"

def _find_col(df, names):
    def norm(s):
        return re.sub(r"[\s_]+", "", str(s)).lower()
    norm_map = {norm(c): c for c in df.columns}
    for n in names:
        key = norm(n)
        if key in norm_map:
            return norm_map[key]
    raise KeyError(f"Could not find any of {names}. Available columns: {list(df.columns)}")

CELLTYPE_COL = _find_col(df_linear_track, ["Cell Type", "Cell type", "cell_type", "celltype"])
GENOTYPE_COL = _find_col(df_linear_track, ["Genotype", "genotype", "geno", "genotype_group", "Genotype group"])

print("Using CELLTYPE_COL =", CELLTYPE_COL)
print("Using GENOTYPE_COL =", GENOTYPE_COL)

def _sig_stars(p):
    if p is None or not np.isfinite(p):
        return "NA"
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"

# numeric metrics to analyze/plot
numeric_cols = df_linear_track.select_dtypes(include=[np.number]).columns.tolist()
cols_to_use = [c for c in numeric_cols if c not in ["tetrode", "cluster", "mouse_name"]]

genotype_levels = sorted(df_linear_track[GENOTYPE_COL].dropna().unique().tolist())
print("Genotypes found:", genotype_levels)

# ---------- 1) 2-way ANOVA within each genotype: Experimenter x Cell type ----------
anova_rows = []
term_exp = f'C(Q("{EXPERIMENTER_COL}"))'
term_ct  = f'C(Q("{CELLTYPE_COL}"))'
term_int = f'{term_exp}:{term_ct}'

for geno in genotype_levels:
    df_g = df_linear_track.loc[df_linear_track[GENOTYPE_COL] == geno]

    for col in cols_to_use:
        test_data = df_g[[col, EXPERIMENTER_COL, CELLTYPE_COL]].dropna()
        if len(test_data) < 10:
            continue
        if test_data[EXPERIMENTER_COL].nunique() < 2 or test_data[CELLTYPE_COL].nunique() < 2:
            continue

        try:
            formula = f'Q("{col}") ~ {term_exp} * {term_ct}'
            model = ols(formula, data=test_data).fit()
            aov = anova_lm(model, typ=2)

            p_exp = float(aov.loc[term_exp, "PR(>F)"]) if term_exp in aov.index else np.nan
            p_ct  = float(aov.loc[term_ct,  "PR(>F)"]) if term_ct  in aov.index else np.nan
            p_int = float(aov.loc[term_int, "PR(>F)"]) if term_int in aov.index else np.nan

            anova_rows.append({
                "genotype": str(geno),
                "column": col,
                "n_observations": int(len(test_data)),
                "n_experimenters": int(test_data[EXPERIMENTER_COL].nunique()),
                "n_cell_types": int(test_data[CELLTYPE_COL].nunique()),
                "p_Experimenter": p_exp,
                "p_CellType": p_ct,
                "p_Interaction": p_int,
                "sig_Experimenter": _sig_stars(p_exp),
                "sig_CellType": _sig_stars(p_ct),
                "sig_Interaction": _sig_stars(p_int),
            })
        except Exception as e:
            print(f"[{geno}] Could not run ANOVA for {col}: {e}")

df_anova_by_genotype = pd.DataFrame(anova_rows)
display_df("2-way ANOVA (within each genotype): Experimenter x Cell type", df_anova_by_genotype)

anova_save_path = os.path.join(
    linear_track_DIR,
    "linear_track_2way_anova_experimenter_celltype_split_by_genotype.csv",
)
df_anova_by_genotype.to_csv(anova_save_path, index=False)
print(f"Saved genotype-split ANOVA table to: {anova_save_path}")

# ---------- 2) Boxplots PDF split by genotype: metric by Cell type (hue=Experimenter) ----------
pdf_path = os.path.join(
    linear_track_DIR,
    "linear_track_boxplots_by_experimenter_and_celltype_split_by_genotype.pdf",
)

with PdfPages(pdf_path) as pdf:
    # overall title page
    fig = plt.figure(figsize=(11.69, 8.27))
    txt = (
        "Open field dataset - Boxplots by Experimenter and Cell type, split by Genotype\n"
        f"Genotype column: {GENOTYPE_COL}\n"
        f"Genotypes: {', '.join(map(str, genotype_levels))}\n"
        "Each genotype section: one page per metric\n"
        "Each page: boxplot (no fliers) + jittered points; x = Cell type, hue = Experimenter\n"
    )
    fig.text(0.05, 0.9, txt, fontsize=14, va="top")
    plt.axis("off")
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)

    for geno in genotype_levels:
        df_g = df_linear_track.loc[df_linear_track[GENOTYPE_COL] == geno]

        exp_levels = sorted(df_g[EXPERIMENTER_COL].dropna().unique().tolist())
        ct_levels = sorted(df_g[CELLTYPE_COL].dropna().unique().tolist())

        # genotype section title page
        fig = plt.figure(figsize=(11.69, 8.27))
        txt = (
            f"{GENOTYPE_COL} = {geno}\n"
            f"Experimenters: {', '.join(map(str, exp_levels)) if exp_levels else 'NA'}\n"
            f"Cell types: {', '.join(map(str, ct_levels)) if ct_levels else 'NA'}\n"
        )
        fig.text(0.05, 0.9, txt, fontsize=16, va="top")
        plt.axis("off")
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

        for col in cols_to_use:
            plot_data = df_g[[col, EXPERIMENTER_COL, CELLTYPE_COL]].dropna()
            if len(plot_data) == 0:
                continue
            if plot_data[CELLTYPE_COL].nunique() < 2:
                continue

            fig = plt.figure(figsize=(11.69, 8.27))
            ax = fig.add_subplot(111)

            sns.boxplot(
                data=plot_data,
                x=CELLTYPE_COL,
                y=col,
                hue=EXPERIMENTER_COL,
                order=ct_levels if len(ct_levels) else None,
                hue_order=exp_levels if len(exp_levels) else None,
                showfliers=False,
                ax=ax,
            )
            sns.stripplot(
                data=plot_data,
                x=CELLTYPE_COL,
                y=col,
                hue=EXPERIMENTER_COL,
                order=ct_levels if len(ct_levels) else None,
                hue_order=exp_levels if len(exp_levels) else None,
                dodge=True,
                jitter=0.2,
                size=2.5,
                alpha=0.35,
                linewidth=0,
                ax=ax,
            )

            # de-duplicate legend (boxplot + stripplot both add)
            handles, labels = ax.get_legend_handles_labels()
            by_label = dict(zip(labels, handles))
            ax.legend(
                by_label.values(),
                by_label.keys(),
                title=EXPERIMENTER_COL,
                bbox_to_anchor=(1.02, 1),
                loc="upper left",
                borderaxespad=0,
            )

            ax.set_title(f"{col} by Cell type (hue=Experimenter) | {GENOTYPE_COL}={geno}")
            ax.set_xlabel(CELLTYPE_COL)
            ax.set_ylabel(col)
            ax.tick_params(axis="x", rotation=45)

            fig.tight_layout()
            pdf.savefig(fig)
            plt.close(fig)

print(f"Saved genotype-split boxplots PDF to: {pdf_path}")

NameError: name 'df_open_field' is not defined